# 1.4 Demo - Advanced Data Quality Checks and Expectations in SDP
**Lesson:** Advanced Data Quality Checks and Expectations in SDP

Row-level `expect` rules are only the start. Production pipelines also need:
1. **Rules as data**: expectations stored in a governed Delta table and loaded at pipeline start (`expect_all*` with a dict)
2. **Zero-data-loss quarantine**: flag rows with `is_quarantined` instead of dropping them, then split into *valid* and *quarantine* datasets
3. **Schema drift detection**: Auto Loader's `_rescued_data` column, monitored by a warn-only expectation
4. **Referential integrity**: a stream-static join to a dimension, with an expectation on the join result
5. **Table-level checks**: aggregate expectations on a materialized view that reconcile row counts across layers
6. **Monitoring**: data quality metrics from the event log

## A. Classroom setup and source data
The US mobile-app orders (JSON) and the product catalog (CSV) from the e-commerce dataset.

In [ ]:
from adelab.classroom import setup
from adelab import datagen
from pyspark.sql import functions as F

DA = setup("1.4", reset=True)
datagen.land_ecommerce(DA.paths.landing, batch=0)
ORDERS = f"{DA.paths.landing}/orders_us"
PRODUCTS = f"{DA.paths.landing}/products"

display(spark.read.json(ORDERS).limit(5))

## B. Rules as data
Keep the rules in a table that data stewards can review and change (with Unity Catalog permissions and history) without touching pipeline code. The pipeline loads the rules for a **tag** when it starts.

Write every rule so it returns `true` or `false`, never `NULL`. `quantity > 0` is `NULL` when `quantity` is missing; Databricks requires constraints to evaluate to true or false and does not document how a `NULL` result is treated (this local engine counts it as a failure). `quantity IS NOT NULL AND quantity > 0` removes the ambiguity.
```python
def get_rules(tag):
    return {r["name"]: r["constraint"]
            for r in spark.read.table(rules_table).filter(F.col("tag") == tag).collect()}
```

In [ ]:
rules = [
    ("valid_order_id", "order_id IS NOT NULL", "orders"),
    ("valid_customer", "customer_id IS NOT NULL", "orders"),
    ("positive_quantity", "quantity IS NOT NULL AND quantity > 0", "orders"),
    ("valid_price", "price IS NOT NULL AND price > 0", "orders"),
]
spark.createDataFrame(rules, "name STRING, constraint STRING, tag STRING") \
     .write.mode("overwrite").saveAsTable("dq_rules")
display(spark.table("dq_rules"))

## C. The pipeline
- `orders_bronze`: Auto Loader with **schema inference + hints** in `rescue` mode. Anything that doesn't fit lands in `_rescued_data`, and the `no_rescued_data` **warn** expectation surfaces drift in the metrics.
- `orders_dq_flagged`: every row, plus `is_quarantined = NOT(rule1 AND rule2 ...)`. It uses `expect_all(rules)` as **warn**, so all rows are kept and every rule is metered.
- `orders_quarantine` / `orders_valid`: the split. The quarantine table also lists which rules each row broke.
- `orders_silver`: valid orders joined to `products`. `known_product` drops orders whose SKU doesn't exist (referential integrity).
- `daily_category_sales` (`expect_or_fail` on revenue) and `dq_reconciliation`: an **aggregate** check proving no row was lost between layers.

![Advanced data quality pipeline: dq_rules is loaded at pipeline start into orders_dq_flagged, which flags every orders_bronze row with is_quarantined; the flag splits rows into the orders_valid view and orders_quarantine; orders_silver joins products and feeds daily_category_sales; dq_reconciliation checks bronze_rows = flagged_rows = valid_rows + quarantined_rows](../images/1.4-rules-quarantine.svg)

**How to read it:**
- The rules never appear in the code: `get_rules("orders")` reads them from `dq_rules` (the table you just created) when the pipeline starts.
- Every row reaches `orders_dq_flagged`; the flag decides between `orders_valid` and `orders_quarantine`, so nothing is dropped silently.
- The bottom band is `dq_reconciliation`, queried in section D. Its two warn expectations are the two equals signs.

In [ ]:
DA.show_source("pipelines/advanced_dq")

In [ ]:
pipe = DA.pipeline("advanced_dq", source="pipelines/advanced_dq", configuration={
    "orders_source": ORDERS, "products_source": PRODUCTS, "rules_table": f"{DA.schema_name}.dq_rules"})
update = pipe.run()

## D. Inspect the results

In [ ]:
display(pipe.expectation_metrics())

In [ ]:
%%sql
SELECT order_id, customer_id, quantity, price, failed_rules FROM orders_quarantine

In [ ]:
%%sql
SELECT * FROM dq_reconciliation

## E. Schema drift arrives
The next batch from the mobile app contains a **new field** (`coupon`) and a **badly typed** value (`quantity: "two"`). Auto Loader keeps the declared schema and moves the surprises into `_rescued_data`, so nothing is lost and the pipeline keeps running.

![Schema drift: one batch 1 order carries a new coupon field and another has quantity "two"; Auto Loader with schema hints in rescue mode writes both to orders_bronze, the second with quantity NULL, and puts the unexpected values in _rescued_data; no_rescued_data counts both rows and the NULL-quantity row goes to orders_quarantine](../images/1.4-schema-drift.svg)

**How to read it:** the middle box is the `cloudFiles` options in `01_bronze.py`. The query below shows the right-hand rows, and the metrics cell after it shows the two outcomes at the bottom: `no_rescued_data` failures (warn, rows kept) and `positive_quantity` failures (quarantined).

In [ ]:
datagen.land_ecommerce(DA.paths.landing, batch=1)
update = pipe.run()

In [ ]:
%%sql
SELECT orderId, quantity, _rescued_data
FROM orders_bronze
WHERE _rescued_data IS NOT NULL

In [ ]:
display(pipe.expectation_metrics().filter("expectation IN ('no_rescued_data', 'positive_quantity')"))

Rows with `quantity = "two"` have `quantity = NULL` in Bronze. `positive_quantity` (`quantity IS NOT NULL AND quantity > 0`) is **false** for them, so they go to quarantine. The raw value is still in `_rescued_data` for repair.

Expected: `no_rescued_data` now reports failed records (the drifted rows of batch 1), and the `positive_quantity` failures include the `"two"` rows.

## F. Change a rule without touching code
The business decides `price` up to 2000 is plausible and anything above is suspicious. Add a rule to the **table** and run again. The new rule applies to new data only. Already-processed rows are not re-evaluated, which is a streaming-table property. Use a full refresh to re-apply rules to history.

In [ ]:
spark.sql("INSERT INTO dq_rules VALUES ('plausible_price', 'price <= 2000', 'orders')")
datagen.land_ecommerce(DA.paths.landing, batch=2)
pipe = DA.pipeline("advanced_dq", source="pipelines/advanced_dq", configuration={
    "orders_source": ORDERS, "products_source": PRODUCTS, "rules_table": f"{DA.schema_name}.dq_rules"})
update = pipe.run()
display(pipe.expectation_metrics().filter("dataset = 'orders_dq_flagged'"))

## G. Data-quality trend from the event log
Every update writes `flow_progress` events. Over time they form a data-quality history you can chart or alert on (Databricks SQL alerts, Lakeview dashboards).

In [ ]:
display(pipe.expectation_metrics(latest_only=False)
        .groupBy("update_id", "expectation")
        .agg(F.sum("failed_records").alias("failed"))
        .orderBy("update_id", "expectation"))

## H. Choosing an expectation action
| Situation | Action |
|---|---|
| You want visibility but must keep every row (Bronze, drift monitoring) | `expect` (warn) |
| Bad rows must never reach consumers, but they are rare and auditable elsewhere | `expect_or_drop` **+ a quarantine table** |
| A violation means the data is unusable or a bug upstream (bad keys, negative totals) | `expect_or_fail` |
| Rules change often or are owned by data stewards | rules table + `expect_all*` |
| Completeness across layers (no lost rows), freshness, volume anomalies | aggregate expectations on a materialized view |

In [ ]:
DA.cleanup()